# Milestone 1 — Dual-T4 TP performance diagnostics

This output-free launcher installs a reviewed source snapshot of the lightweight SDK, previews the controlled offline-engine matrix, then runs it on Kaggle T4 x2. It does not contain benchmark results, upload evidence, regenerate Qwen, or claim PCIe/NCCL causality. Configure the exact source and existing Qwen input paths before execution.

In [ ]:
from pathlib import Path

SOURCE_ROOT = Path('/kaggle/input/kaggle-vllm-milestone-1-source/kaggle-vllm')
SOURCE_IDENTITY = 'REPLACE_WITH_REVIEWED_GIT_COMMIT'
QWEN_MODEL = Path('/kaggle/input/kaggle-vllm-models/qwen2.5-3b-t4x2-sharded')
SDK_TARGET = Path('/kaggle/working/kaggle-vllm-milestone-1-sdk')
EVIDENCE_DIR = Path('/kaggle/working/kaggle-vllm-tp-milestone-1')

assert (SOURCE_ROOT / 'pyproject.toml').is_file(), SOURCE_ROOT
assert (SOURCE_ROOT / 'scripts/kaggle_tp_diagnostics.py').is_file()
assert SOURCE_IDENTITY != 'REPLACE_WITH_REVIEWED_GIT_COMMIT'
assert QWEN_MODEL.is_dir(), QWEN_MODEL
assert not SDK_TARGET.exists(), f'refusing existing SDK target: {SDK_TARGET}'
assert not EVIDENCE_DIR.exists(), f'refusing existing evidence dir: {EVIDENCE_DIR}'

In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--target', str(SDK_TARGET), str(SOURCE_ROOT)],
    check=True,
)
RUN_ENV = dict(os.environ)
RUN_ENV['PYTHONPATH'] = str(SDK_TARGET)
RUNNER = SOURCE_ROOT / 'scripts/kaggle_tp_diagnostics.py'

## Audit the plan

The dry run performs no bootstrap, GPU work, model download, or evidence-directory creation. Review the printed model identities, TP degrees, engine settings, workloads, and destinations before continuing.

In [ ]:
base_command = [
    sys.executable, str(RUNNER),
    '--source-identity', SOURCE_IDENTITY,
    '--expected-sdk-version', '0.2.0',
    '--qwen-model', str(QWEN_MODEL),
    '--output-dir', str(EVIDENCE_DIR),
]
subprocess.run([*base_command, '--dry-run'], check=True, env=RUN_ENV)

## Execute on Kaggle GPU T4 x2

The runner strictly validates the documented profile, bootstraps and activates the immutable native runtime, uses isolated child processes for each engine configuration, and writes JSON/log/topology/checksum evidence only under `/kaggle/working`. It never uploads evidence automatically.

In [ ]:
subprocess.run(base_command, check=True, env=RUN_ENV)
print((EVIDENCE_DIR / 'summary.json').read_text())
print((EVIDENCE_DIR / 'SHA256SUMS.txt').read_text())